# Part 1: RAG in Databricks

## Section 1: Environment Setup

First we set up the Unity Catalog infrastructure and upload the 3 RAG research papers to a Volume.

Fill in and Run all cells in order.

In [0]:
# %load_ext autoreload
# %autoreload 2

In [0]:
# %pip install --quiet databricks-ai-search matplotlib databricks_langchain langchain_core langchain_openai umap-learn plotly

In [0]:
# dbutils.library.restartPython()

In [0]:
# spark.sql("DROP CATALOG IF EXISTS cs4603 CASCADE")

### What is Unity Catalog?

Unity Catalog is Databricks' centralized governance layer for data and AI assets. It organizes everything into a three-level namespace:

- **Catalog:** the top-level container, like a database server. Separates environments (e.g., `dev`, `prod`) or projects.
- **Schema:** a namespace within a catalog. Groups related tables, views, and volumes.
- **Volume:** a mount point for files (PDFs, images, CSVs) stored in cloud storage. Tables can reference files in a volume via `READ_FILES()`.

The hierarchy is: `catalog.schema.volume` or `catalog.schema.table`.

### Create Infrastructure

In [0]:
from databricks.ai_search.client import AISearchClient
from databricks_langchain import DatabricksVectorSearch, ChatDatabricks
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# TODO: Create catalog, schema, and volume
# Hint: SEE Tutorial

CATALOG = ...    
SCHEMA = ...    
VOLUME = ...   
CHUNK_TABLE = f"{CATALOG}.{SCHEMA}.chunks"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
PARSE_TABLE = f"{CATALOG}.{SCHEMA}.parsed_documents"

VS_ENDPOINT = ...   # e.g. f"{CATALOG}_rag_endpoint"
VS_INDEX = ...      # e.g. f"{CATALOG}.{SCHEMA}.rag_index"
EMBEDDING_MODEL = "..." # see .env
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

In [0]:
notebook_dir = # get current dir
pdf_files = [f for f in ... if f.lower().endswith(...)]

for ... in ...:
  src = ...
  dst = ...
  # dbutils, copy src to dst
  print(f"  Uploaded: {pdf}")

print("\nUpload complete.")

### Verify

**Observe:** Is your catalog listed? Is rag_data schema shown? Is corpus volume shown? 

`DESCRIBE` your catalog. (Name, comment, owner, Type)

In [0]:
display(spark.sql(f"..."))

In [0]:
display(spark.sql(f"..."))

In [0]:
display(spark.sql(f"..."))

In [0]:
display(spark.sql(f"..."))

## Section 2: PDF Parsing with `ai_parse_document`

This section parses the 3 RAG research papers into structured elements using Databricks' `ai_parse_document` function.

### What is `ai_parse_document`?

`ai_parse_document` takes a binary file (PDF, DOCX, image) and returns a **VARIANT**: a structured JSON object containing:

- **`document.elements`**: every text block, table, figure, heading, and caption, each with a `type`, `content`, and `confidence` score
- **`document.pages`**: page-level metadata (page IDs, image URIs)
- **`metadata`**: file-level info (path, size, modification time)

> Read more: [ai_parse_document](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_parse_document)


### [EXPLORATION]

What was the previous API way to achieve this? What information does this version give that the previous version doesn't?

*Hint: the old approach used `ai_extract` to pull specific fields from a STRING-stored parse result.*

### Parse PDFs

Create the `PARSE_TABLE` with path (STRING) and parsed (VARIANT) fields. 

Add `TBLPROPERTIES (delta.enableChangeDataFeed = true)`, we need it for later

Use `ai_parse_document` to `INSERT` the files you read into the `PARSE_TABLE`

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {PARSE_TABLE} (
        ...
    ) TBLPROPERTIES (delta.enableChangeDataFeed = ...)
""")

spark.sql(f"""
    INSERT OVERWRITE {PARSE_TABLE}
    SELECT
        ...
    FROM ...
""")

row_count = spark.table(PARSE_TABLE).count()
print(f"Parsed {row_count} document(s) into '{PARSE_TABLE}'.")

### Explore the VARIANT Output

Let's look at what `ai_parse_document` returns. We use `to_json()` to convert the VARIANT to a readable string.

**Observe:** What element types exist? How many of each?

In [0]:
raw = spark.sql(f"""
    SELECT ... AS raw_json
    FROM {PARSE_TABLE}
    LIMIT 1
""").collect()[0]["raw_json"]

print(raw[:2000])
print(f"\n... ({len(raw)} total chars)")

### Flatten Elements

Each document contains an array of elements. We use `LATERAL variant_explode()` to unwrap this array into individual rows, one row per element.

**Observe:** How many rows are produced? One per element across all PDFs?

> Read more: [variant_explode](https://docs.databricks.com/aws/en/sql/language-manual/functions/variant_explode)

In [0]:
elements_df = spark.sql(f"""
    SELECT
        path,
        # TODO: extract element fields from parsed (hint: nested JSON — use element.value:<field>::<TYPE>)
    FROM {PARSE_TABLE},
        ... AS element
""")

print(f"Total elements: {elements_df.count()}")
display(elements_df.limit(10))

### Element Type Breakdown

**Observe:** Which PDF has the most tables? The most text elements?

In [0]:
display(
    spark.sql(f"""
        SELECT
            ... AS element_type,
            COUNT(*) AS count
        FROM {PARSE_TABLE},
            ...
        GROUP BY ... -- hint: group by element type
        ORDER BY ... -- hint: order by count descending
    """)
)

In [0]:
display(
    spark.sql(f"""
        SELECT
            ...,
            ... AS element_type,
            COUNT(*) AS count
        FROM {PARSE_TABLE},
            ...
        GROUP BY ... -- hint: group by path and element type
        ORDER BY ... -- hint: order by path, then count descending
    """)
)

## Section 3: Chunking with `ai_prep_search`

Next we chunk the parsed PDF elements into searchable units using `ai_prep_search`, and explore how enrichment improves retrieval.

### What is `ai_prep_search`?

`ai_prep_search` takes the VARIANT output from `ai_parse_document` and produces **semantic chunks**. Semantic chunking builds upon sentenced based chunking by incorporating *semantic* relevance. This ensures the document is split based on *meaning* and *structure* rather than arbitrary character limits. Each chunk comes with two versions of the text:

- **`chunk_to_retrieve`:** the raw text (what you'd show the user)
- **`chunk_to_embed`:** enriched text with metadata + LLM context (what gets embedded for search)

> Read more: [ai_prep_search](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_prep_search)

### Chunk with `ai_prep_search`

Create a `CHUNK_TABLE` with fields:
- chunk_id
- chunk_to_retrieve
- chunk_to_embed
- source_uri

All strings

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CHUNK_TABLE} (
        ...
    ) TBLPROPERTIES (delta.enableChangeDataFeed = ...)
""")

spark.sql(f"""
    INSERT OVERWRITE {CHUNK_TABLE}
    SELECT
        ... -- hint: extract chunk fields from variant_explode result
    FROM (
        SELECT path, ... AS result FROM {PARSE_TABLE}
    ) prepped,
        ... AS chunk
""")

chunk_count = spark.table(CHUNK_TABLE).count()
print(f"Created {chunk_count} chunks in '{CHUNK_TABLE}'.")

### Chunk Counts per Document

**Observe:** How many chunks per document?

In [0]:
display(
    spark.sql(f"""
        SELECT
            source_uri,
            COUNT(*) AS chunk_count
        FROM {CHUNK_TABLE}
        GROUP BY ...
        ORDER BY ...
    """)
)

### Raw vs Enriched Text

`chunk_to_retrieve` is the raw text. `chunk_to_embed` wraps that text with document-level context: metadata, LLM-discovered fields, and related questions. The enriched version is what gets embedded, so the embedding captures more meaning.

**Observe:** What extra information does `chunk_to_embed` contain?

Let's see the difference side by side.

In [0]:
display(
    spark.sql(f"""
        SELECT
            ...
        FROM {CHUNK_TABLE}
        LIMIT 3
    """)
)

[GRADED QUESTION]

### Q1.
Why would the enriched version produce better search results?

[EXPLORATION]

What are the 3-4 most important enrichments/metadata that `ai_prep_search` adds to `chunk_to_embed`?

### Size Distribution

**Observe:** What is the average chunk size? Is the distribution skewed?

In [0]:
import matplotlib.pyplot as plt

sizes = spark.sql(f"""
    SELECT ... AS chunk_size
    FROM {CHUNK_TABLE}
""").toPandas()["chunk_size"].tolist()

plt.figure(figsize=(10, 4))
plt.hist(sizes, bins=30, edgecolor="black", alpha=0.7)
plt.xlabel("Chunk Size (characters)")
plt.ylabel("Count")
plt.title("Distribution of Chunk Sizes (chunk_to_retrieve)")
plt.tight_layout()
plt.show()

print(f"Chunks: {len(sizes)}")
print(f"Mean size: {sum(sizes)/len(sizes):.0f} chars")
print(f"Min: {min(sizes)}, Max: {max(sizes)}")

In [0]:
import matplotlib.pyplot as plt

sizes = spark.sql(f"""
    SELECT ... AS chunk_size
    FROM {CHUNK_TABLE}
""").toPandas()["chunk_size"].tolist()

plt.figure(figsize=(10, 4))
plt.hist(sizes, bins=30, edgecolor="black", alpha=0.7)
plt.xlabel("Chunk Size (characters)")
plt.ylabel("Count")
plt.title("Distribution of Chunk Sizes (chunk_to_embed)")
plt.tight_layout()
plt.show()

print(f"Chunks: {len(sizes)}")
print(f"Mean size: {sum(sizes)/len(sizes):.0f} chars")
print(f"Min: {min(sizes)}, Max: {max(sizes)}")

### [GRADED QUESTION]

Why does `chunk_to_embed` contain more text than `chunk_to_retrieve`?

## Section 4: Vector Search Index

Next we create a vector search index over the chunks table. 

> Note: Databricks manages the embeddings and the ANN index automatically. This will NOT be the case in your local implementation.

### What is a Vector Search Index?

A vector search index stores embeddings (numeric vectors) for each row in a Delta table and enables fast approximate nearest neighbor (ANN) search at query time.

A **Delta Sync index** keeps itself in sync with the source Delta table. When new rows are inserted or updated, the index recomputes embeddings and updates the ANN structure automatically.

> Read more: [Create AI Search Index](https://docs.databricks.com/aws/en/ai-search/create-ai-search)

### Endpoint Types [GRADED QUESTION]

Databricks supports two endpoint types:

- **STANDARD** 
- **STORAGE_OPTIMIZED** 

What are their trade-offs?

### Create the Endpoint

In [0]:
client = AISearchClient()

# HINT: Tutorial
existing = [e["name"] for e in client.list_endpoints().get("endpoints", [])]
if ... not in ...:
    ...
    print(f"Created endpoint '{VS_ENDPOINT}'")
else:
    print(f"Endpoint '{VS_ENDPOINT}' already exists.")

### Create the Delta Sync Index

In [0]:
existing_indexes = [idx["name"] for idx in client.list_indexes(name=VS_ENDPOINT).get("vector_indexes", [])]
if VS_INDEX not in existing_indexes:
    ...
    print(f"Created index '{VS_INDEX}'")
else:
    print(f"Index '{VS_INDEX}' already exists.")

### `Describe` the Index

**Observe:** What is the endpoint type? Embedding model? Is status READY? What does the message say?

In [0]:
index = client.get_index(endpoint_name=..., index_name=...)
...

### [EXPLORATORY]

We used `databricks-qwen3-embedding-0-6b`. The Databricks catalog also lists `databricks-gte-large-en` as an available embedding model. 

How do embedding models affect the index? similarity search? And overall, retrieval system?

### Verify the Index is Queryable

**Observe:** Does the query return results? What does this confirm?

In [0]:
test = index.similarity_search(
    query_text="What is RAG?",
    columns=[...],
    num_results=1,
)
if test.get("result", {}).get("data_array"):
    print("Index is queryable.")
else:
    print("Index not ready yet, wait a minute and re-run.")

### Idempotent Ingestion

For vector search, idempotent ingestion is handled by Delta Sync: the index manager deduplicates by primary key when syncing from the Delta table. This means you can re-run the ingestion pipeline without creating duplicate entries in the search index.

In your local implementation (Part 2), you will use `SQLRecordManager` to achieve the same: a sidecar `upsertion_record` table tracking content hashes, enabling `index()` to skip unchanged chunks and delete stale ones based on cleanup mode (`None`, `"incremental"`, `"full"`).

### [GRADED QUESTION]

What algorithm does a STANDARD endpoint use for ANN search?

### [GRADED QUESTION]

What is the difference between IVFFlat and HNSW?

### [EXPLORATION]

#### When would you choose STORAGE_OPTIMIZED over STANDARD?

Think about your use case: production RAG system serving thousands of queries per day. When might the higher latency of STORAGE_OPTIMIZED be acceptable?

### [EXPLORATION]

#### Metadata Filtering

The chunks table has `parse_method` metadata. When would filtering by metadata improve retrieval results? For example, if a question is specifically about PDF content versus HTML content.

## Section 5: Retrieval & Similarity Search

Now we query the index using three different retrieval methods and compare the results.

### Connect to the Index

In [0]:
vectorstore = DatabricksVectorSearch(
    index_name=...,
    columns=[...],
)

### Retrieval Queries

Run 3 queries using semantic retrieval.

**Observe:** How similar are the results across the 3 queries? Do they all retrieve relevant chunks?

In [0]:
queries = [
    "What is retrieval-augmented generation?",
    "How does RAPTOR improve retrieval?",
    "What are the limitations of dense retrieval?",
]

for q in queries:
    results = ...  # Hint: look at vectorstore methods, with score
    print(f"\nQ: {q}")
    for i, (doc, score) in enumerate(results, 1):
        print(f"  {i}. {doc.page_content[:150]}... (score: {score})")

### What is FULL_TEXT and HYBRID Search?

Databricks supports three retrieval methods:

- **ANN** (default): semantic similarity via embedding vectors. Best for conceptual queries.
- **FULL_TEXT**: keyword matching (BM25-style). Best for exact terms, product IDs, specific identifiers.
- **HYBRID**: combines ANN + FULL_TEXT using **Reciprocal Rank Fusion (RRF)**. Merges both ranked lists into a single ordering:

`RRF_score(d) = SUM( 1 / (k + rank_i(d)) )` for each method `i`, where `k = 60` (smoothing constant).

> Read more: [Query an AI Search index](https://docs.databricks.com/aws/en/vector-search/query-vector-search)

**Observe:** How do FULL_TEXT and HYBRID results differ from ANN?

In [0]:
query = "What are the limitations of dense retrieval?"

# FULL_TEXT (keyword only)
# get the index
ft_response = index.similarity_search(
    query_text=query,
    columns=[...],
    num_results=3,
    query_type="..."
)
print("=== FULL_TEXT (keyword only) ===")
for i, row in enumerate(ft_response.get("result", {}).get("data_array", []), 1):
    score = row[2] if len(row) > 2 else None
    print(f"  {i}. {row[0][:150]}... (score: {score})")

print()

# HYBRID (semantic + keyword via RRF)
hybrid_response = index.similarity_search(
    query_text=query,
    columns=[...],
    num_results=3,
    query_type="..."
)
print("=== HYBRID (semantic + keyword) ===")
for i, row in enumerate(hybrid_response.get("result", {}).get("data_array", []), 1):
    score = row[2] if len(row) > 2 else None
    print(f"  {i}. {row[0][:150]}... (score: {score})")

How do the results differ across the 3 retrieval methods?

[GRADED QUESTION]

### Q5.

In 1 line ONLY, describe what kind of query each method would be best at.

[GRADED QUESTION]

### Q6.

Are the similarity scores across FULL_TEXT, HYBRID, and ANN comparable? Why or why not? 1 line answer ONLY.

[EXPLORATION]

### What if the retriever returns no documents?

How would you handle the case where the retriever returns zero results? Think about what the chain would do and how you might add a fallback.

[EXPLORATION]

### Why k=4?

Our retriever uses `k=4`. What happens with k=1 versus k=10? Would ranking the documents solve the issue?

## Section 6: LCEL RAG Chain

Next we build a complete RAG chain using LangChain Expression Language (LCEL). This is the exact pattern you will implement locally in Part 3.

### What is LCEL?

LangChain Expression Language (LCEL) chains runnables together using the pipe operator (`|`). Each component is a **Runnable** that transforms data and passes it to the next. The chain is invoked with a single `.invoke()` call.

The RAG chain topology:

```
input question
    +-- "question" -> RunnablePassthrough (pass-through)
    +-- "context"  -> retriever -> format_docs (join into string)
           | (both feed into)
    rag_prompt (system + human template)
           |
    ChatDatabricks (LLM)
           |
    StrOutputParser (extract string)
```

### What is format_docs?

`format_docs` converts a list of LangChain `Document` objects into a single string that fits into the prompt template. It gives the RAG engineer full control over exactly what context the LLM sees: numbering, truncation, metadata stamping, filtering.

Since Databricks `ai_prep_search` chunks already contain enriched metadata (headers, page numbers, context sentences), `format_docs` here is simple: just `[1] <text>`. In your local implementation (Part 3), `format_docs` will need to extract and format source metadata (filename, page number) manually, since local chunks are plain text.

> Read more: [LangChain + Databricks VS](https://docs.langchain.com/oss/python/integrations/vectorstores/databricks_vector_search)

### Define the Retriever

In [0]:
retriever = ...  # k=4

### Inspect Retrieved Documents

**Observe:** What metadata is returned with each chunk?

In [0]:
docs = retriever.invoke("What is RAG?")[:5]
for i, doc in enumerate(docs, 1):
    print(f"--- Chunk {i} ---")
    print(f"Metadata: {doc.metadata}")
    print(f"Text: {doc.page_content[:150]}...")
    print()

### Define format_docs

In [0]:
def format_docs(docs):
    return "\n\n".join(
        f"..."
        for i, doc in enumerate(docs)
    )

### Define the Prompt

In [0]:
rag_prompt = ...  # Hint: Tutorial

### Build the Chain

In [0]:
llm = ...

rag_chain = ...

print("RAG chain ready.")

### Ask Questions

**Observe:** Are the answers grounded in the retrieved context? Do the [1], [2] citations match the chunks?

In [0]:
questions = [
    "What is retrieval-augmented generation?",
    "How does RAPTOR improve retrieval?",
    "What are the key components of a RAG pipeline?",
]

for q in questions:
    print(f"\nQ: {q}")
    answer = ...  # invoke your chain
    print(f"A: {answer}")
    print("-" * 60)

[GRADED QUESTION]

### Q7.

What does `format_docs` enable a RAG engineer to do? Is the upside worth the engineering effort? Can you say the same after implementing it in Part 3?

[EXPLORATION]

### Hallucination Safeguards

Our prompt says "say you don't know if not in context." Is that enough? Could the LLM still hallucinate despite this instruction? What additional safeguards would you add?

[EXPLORATION]

### Cost Management

More chunks and a bigger LLM both increase costs. Is it possible to reasonably maintain costs while increasing both?

## EXTRA: Section 7 Understanding Embedding Space

Visualize how chunk embeddings cluster in 2D space. This helps you understand why retrieval works and how chunking and embedding choices affect retrieval quality.

### What is UMAP?

UMAP (Uniform Manifold Approximation and Projection) reduces high-dimensional embedding vectors to 2D for visualization. UMAP captures non-linear relationships and preserves local neighborhood structure: chunks that are close in high-dimensional space stay close in 2D.

This makes UMAP useful for inspecting whether chunks from the same document cluster together, and whether semantically similar chunks from different documents overlap.

### Load Chunks

**Observe:** How many chunks per source? Do different documents have very different chunk counts?

In [0]:
import pandas as pd

df = spark.sql(f"""
    SELECT chunk_id, chunk_to_retrieve, source_uri
    FROM {CHUNK_TABLE}
""").toPandas()

print(f"Total chunks: {len(df)}")
display(df["source_uri"].value_counts())

### Compute Embeddings

We use the same embedding model that powers our vector search index (`databricks-qwen3-embedding-0-6b`).

In [0]:
from databricks_langchain import DatabricksEmbeddings
import numpy as np

embeddings_model = DatabricksEmbeddings(endpoint=EMBEDDING_MODEL)

texts = df["chunk_to_retrieve"].tolist()

# Filter and validate: ensure all are non-empty strings
texts = [str(t).strip() for t in texts if t and str(t).strip()]
print(f"Validated texts count: {len(texts)} out of {len(df)}")

vectors = embeddings_model.embed_documents(texts)
embeddings = np.array(vectors)

print(f"Embeddings shape: {embeddings.shape}")

### UMAP Reduction

In [0]:
import umap

reducer = umap.UMAP(n_components=2, random_state=42)
coords_2d = reducer.fit_transform(embeddings)

print(f"Reduced shape: {coords_2d.shape}")

### Visualize

**Observe:** Do chunks from the same document cluster together? Are there overlapping regions between documents?

In [0]:
import plotly.express as px

df["x"] = coords_2d[:, 0]
df["y"] = coords_2d[:, 1]

fig = px.scatter(
    df,
    x="x",
    y="y",
    color="source_uri",
    title="Chunk Embeddings by Source Document",
    hover_data=["chunk_id"],
)
fig.show()

[EXPLORATION]

### Chunk Size Effect

If you re-ran `ai_prep_search` with smaller chunks, how would the UMAP visualization change? Think about what happens to embedding density and cluster boundaries.

[EXPLORATION]

### Embedding Model Choice

Our index uses `databricks-qwen3-embedding-0-6b`. How might switching to a different model (e.g., `databricks-gte-large-en` or `databricks-bge-large-en`) change the cluster structure in the visualization? What tradeoffs would that introduce?